In [1]:
from pyvirtualdisplay import Display
import pyvista as pv

# 1. Start the virtual X-server
# This creates a dummy display that VTK will happily bind to.
# Because /host/usr/lib is mapped, it will still use GPU.
display = Display(visible=0, size=(1920, 1080))
display.start()

# 2. Instantiate Plotter
# Now that a DISPLAY variable exists, the X11 factory will succeed
# and use the virtual display we just created.
plotter = pv.Plotter(off_screen=True)

In [2]:
import numpy as np
from mpi4py import MPI
from dolfinx import mesh, fem, io, default_scalar_type, plot
from dolfinx.fem.petsc import LinearProblem
import ufl


# Create mesh and define function space
domain = mesh.create_unit_square(MPI.COMM_WORLD, 8, 8, mesh.CellType.triangle)
domain.topology.create_connectivity(domain.topology.dim - 1, domain.topology.dim)
V = fem.functionspace(domain, ("Lagrange", 1))

# Define boundary condition
boundary_facets = mesh.exterior_facet_indices(domain.topology)
boundary_dofs = fem.locate_dofs_topological(V, domain.topology.dim - 1, boundary_facets)
bc = fem.dirichletbc(default_scalar_type(0.0), boundary_dofs, V)

# Define variational problem
u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)
f = fem.Constant(domain, default_scalar_type(1.0))
a = ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx
L = f * v * ufl.dx

# Compute solution
problem = LinearProblem(a, L, bcs=[bc], 
                        petsc_options={"ksp_type": "preonly", "pc_type": "lu"},
                        petsc_options_prefix="icarus_solver")
uh = problem.solve()

# Rendering
topology, cell_types, geometry = plot.vtk_mesh(V)
grid = pv.UnstructuredGrid(topology, cell_types, geometry)
grid.point_data["u"] = uh.x.array.real

plotter = pv.Plotter()
plotter.add_mesh(grid, show_edges=True, cmap="viridis")
plotter.view_xy()
plotter.screenshot("solution.png")
plotter.close()
print("Saved artifacts using hardware acceleration.")

Saved artifacts using hardware acceleration.


In [3]:
import numpy as np
from mpi4py import MPI
from dolfinx import mesh, fem, plot
from dolfinx.fem.petsc import LinearProblem
import ufl
import pyvista as pv
from openmdao.api import ExplicitComponent

class FEniCSComponent(ExplicitComponent):
    def setup(self):
        self.add_input('f', val=1.0)
        self.add_output('solution', val=0.0)
        # Initialize domain once if constant
        self.domain = mesh.create_unit_square(MPI.COMM_WORLD, 8, 8, mesh.CellType.triangle)
        self.V = fem.functionspace(self.domain, ("Lagrange", 1))

    def compute(self, inputs, outputs):
        f_val = inputs['f'][0]
        
        # Define Variational Problem
        u = ufl.TrialFunction(self.V)
        v = ufl.TestFunction(self.V)
        f = fem.Constant(self.domain, f_val)
        a = ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx
        L = f * v * ufl.dx
        
        # Dirichlet BC
        boundary_facets = mesh.exterior_facet_indices(self.domain.topology)
        boundary_dofs = fem.locate_dofs_topological(self.V, self.domain.topology.dim - 1, boundary_facets)
        bc = fem.dirichletbc(0.0, boundary_dofs, self.V)
        
        problem = LinearProblem(a, L, bcs=[bc], petsc_options={"ksp_type": "preonly", "pc_type": "lu"})
        uh = problem.solve()

        # Rendering
        topology, cell_types, geometry = plot.vtk_mesh(self.V)
        grid = pv.UnstructuredGrid(topology, cell_types, geometry)
        grid.point_data["u"] = uh.x.array.real
        
        # Plotter defaults to EGL automatically now
        plotter = pv.Plotter(off_screen=True)
        plotter.add_mesh(grid, cmap="viridis")
        plotter.screenshot(f"opt_step_{f_val:.2f}.png")
        plotter.close()

        outputs['solution'] = uh.x.array.mean()